# Notebook 3: Lagrangian Duality in SVM

**Difficulty:** ⭐⭐⭐⭐⭐ | **Estimated Time:** 3 hours | **Week 10 — Support Vector Machines & Kernel Methods**

---

## Why This Matters

Lagrangian duality is the mathematical engine that makes SVMs practical and powerful. Without it:
- We cannot efficiently solve SVMs on high-dimensional data (text, images, genomics)
- The kernel trick — which lets SVMs handle non-linear problems — would be impossible
- We would have no principled way to identify which training points actually determine the decision boundary

Understanding duality transforms SVM from a black box into a transparent, interpretable system.

---

## Real-World Analogy First: Bidding on a House

Imagine you want to buy a house at minimum cost, but you have constraints:
- Must have at least 3 bedrooms
- Must be within 10 km of work
- Must have a garage

**Primal approach (direct):** Browse every listing, filter by all constraints, find the cheapest one that satisfies all of them. This is straightforward but you must evaluate every listing.

**Dual approach (indirect):** Instead of checking constraints directly, attach a *price tag* (Lagrange multiplier) to each constraint. Ask: "What's the maximum lower bound on the house price, given I penalize constraint violations?" Both approaches give the **same optimal price** (this is *strong duality*), but the dual tells you *which constraints are binding* — a constraint is binding if its multiplier is non-zero.

In SVM terms:
- **Binding constraints** = support vectors (the training points that define the margin)
- **Non-binding constraints** = interior points (αᵢ = 0, do not affect the boundary)
- The Lagrange multipliers αᵢ reveal WHICH points matter

---

## Context: Email Spam Classification

We classify emails using two features:
- **word_count**: total number of words in the email
- **spam_word_ratio**: fraction of words that are common spam words ("free", "win", "click", etc.)

Labels: +1 = spam, -1 = ham (legitimate)

## Section 1: The Primal SVM Problem — Plain English

The hard-margin SVM primal problem is:

$$\min_{w, b} \frac{1}{2}\|w\|^2 \quad \text{subject to} \quad y_i(w \cdot x_i + b) \geq 1 \quad \forall i$$

**In plain English:** Find the hyperplane that separates spam from ham with the **widest possible margin** (maximizing margin = minimizing ‖w‖²), while ensuring every training point is on the correct side with a gap of at least 1.

This is a **Quadratic Program (QP)**: quadratic objective, linear constraints. It has `p + 1` variables (the `p` components of `w` and the bias `b`).

## Section 2: The Lagrangian — Turning Constraints into Penalties

For each constraint $g_i(w) \leq 0$ where $g_i = 1 - y_i(w \cdot x_i + b)$, we introduce a **Lagrange multiplier** $\alpha_i \geq 0$.

The Lagrangian is:

$$L(w, b, \alpha) = \frac{1}{2}\|w\|^2 - \sum_i \alpha_i \left[y_i(w \cdot x_i + b) - 1\right]$$

**Intuition:** If a constraint is violated ($y_i(w \cdot x_i + b) < 1$), the term inside the brackets is negative, so subtracting it *increases* the Lagrangian — penalizing the violation.

## Section 3: Deriving the Dual

Take partial derivatives and set to zero:

**Step 1:** $\frac{\partial L}{\partial w} = 0 \Rightarrow w = \sum_i \alpha_i y_i x_i$

**Step 2:** $\frac{\partial L}{\partial b} = 0 \Rightarrow \sum_i \alpha_i y_i = 0$

**Step 3:** Substitute back into $L$ — the $w$ and $b$ terms cancel, giving the **dual objective**:

$$\max_{\alpha} \sum_i \alpha_i - \frac{1}{2} \sum_i \sum_j \alpha_i \alpha_j y_i y_j (x_i \cdot x_j)$$

subject to: $\alpha_i \geq 0$ and $\sum_i \alpha_i y_i = 0$

**This is also a QP, but now in $\alpha$-space** — `n` variables (one per training point) instead of `p+1`.

## Section 4: KKT Complementary Slackness

The KKT conditions include the crucial **complementary slackness** condition:

$$\alpha_i \left[y_i(w \cdot x_i + b) - 1\right] = 0 \quad \forall i$$

This means **exactly one** of the following holds for each training point:
- $\alpha_i = 0$: the point is NOT a support vector (it's far from the margin, doesn't affect $w$)
- $y_i(w \cdot x_i + b) = 1$: the point IS on the margin — it IS a support vector

**This is why SVMs are sparse:** most $\alpha_i = 0$, and only the support vectors (a small subset) define the decision boundary.

In [ ]:
# ─── Imports ───────────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from scipy.optimize import minimize
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler

np.random.seed(42)
sns.set_theme(style='whitegrid')

print("All libraries loaded successfully.")
print(f"NumPy: {np.__version__}")

In [ ]:
# ─── Section 1: Generate Linearly Separable 2D Spam Dataset ───────────────────
np.random.seed(42)

n_spam = 50
n_ham  = 50

# Spam emails: high word count, high spam_word_ratio
spam_wc    = np.random.normal(loc=300, scale=40, size=n_spam)   # word_count
spam_swr   = np.random.normal(loc=0.70, scale=0.06, size=n_spam) # spam_word_ratio

# Ham emails: lower word count, lower spam_word_ratio
ham_wc     = np.random.normal(loc=150, scale=40, size=n_ham)
ham_swr    = np.random.normal(loc=0.20, scale=0.06, size=n_ham)

# Clip to realistic ranges
spam_wc  = np.clip(spam_wc,  50, 600)
spam_swr = np.clip(spam_swr, 0.01, 0.99)
ham_wc   = np.clip(ham_wc,   50, 600)
ham_swr  = np.clip(ham_swr,  0.01, 0.99)

X_raw = np.vstack([
    np.column_stack([spam_wc, spam_swr]),
    np.column_stack([ham_wc,  ham_swr])
])
y = np.hstack([np.ones(n_spam), -np.ones(n_ham)])

# Standardize features — important for SVM
scaler = StandardScaler()
X = scaler.fit_transform(X_raw)

# Build a DataFrame for inspection
df = pd.DataFrame({
    'word_count_raw':      X_raw[:, 0],
    'spam_word_ratio_raw': X_raw[:, 1],
    'word_count_scaled':   X[:, 0],
    'spam_ratio_scaled':   X[:, 1],
    'label':               y,
    'class':               ['Spam' if yi == 1 else 'Ham' for yi in y]
})

print(f"Dataset shape: {X.shape}")
print(f"Classes: {np.unique(y)} (Spam=+1, Ham=-1)")
print(f"\nFirst 5 rows:")
df.head()

In [ ]:
# ─── Visualize the raw dataset ─────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

colors = {'Spam': '#e74c3c', 'Ham': '#2ecc71'}

for cls, grp in df.groupby('class'):
    axes[0].scatter(grp['word_count_raw'], grp['spam_word_ratio_raw'],
                    c=colors[cls], label=cls, alpha=0.7, edgecolors='k', linewidths=0.4, s=60)
axes[0].set_xlabel('Word Count', fontsize=12)
axes[0].set_ylabel('Spam Word Ratio', fontsize=12)
axes[0].set_title('Raw Features — Email Dataset', fontsize=13, fontweight='bold')
axes[0].legend(fontsize=11)

for cls, grp in df.groupby('class'):
    axes[1].scatter(grp['word_count_scaled'], grp['spam_ratio_scaled'],
                    c=colors[cls], label=cls, alpha=0.7, edgecolors='k', linewidths=0.4, s=60)
axes[1].set_xlabel('Word Count (Scaled)', fontsize=12)
axes[1].set_ylabel('Spam Word Ratio (Scaled)', fontsize=12)
axes[1].set_title('Standardized Features (Input to SVM)', fontsize=13, fontweight='bold')
axes[1].legend(fontsize=11)

plt.suptitle('Email Spam Dataset — Linearly Separable', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()
print("The two classes are well-separated — a hard-margin SVM should work perfectly.")

In [ ]:
# ─── Section 2: Dual SVM Implementation from Scratch ─────────────────────────

def dual_svm(X, y, tol=1e-5):
    """
    Solve the SVM dual problem using scipy.optimize.minimize (SLSQP).

    Dual objective (maximization form):
        max  Σ αᵢ − (1/2) Σᵢ Σⱼ αᵢ αⱼ yᵢ yⱼ (xᵢ·xⱼ)
    Equivalently (minimization form for scipy):
        min  (1/2) αᵀ K α − Σ αᵢ
    where K[i,j] = yᵢ yⱼ (xᵢ·xⱼ)   (the modified Gram matrix)

    Constraints:
        αᵢ ≥ 0  (bounds)
        Σᵢ αᵢ yᵢ = 0  (equality constraint)

    Returns: w, b, alphas
    """
    n = len(X)

    # Modified Gram matrix K[i,j] = yᵢ yⱼ (xᵢ · xⱼ)
    # Efficient computation: K = (y * X) @ (y * X)ᵀ
    yX = y[:, None] * X          # shape (n, p)
    K  = yX @ yX.T               # shape (n, n)

    # Objective: (1/2) αᵀ K α − 1ᵀα  (we minimize, so negate the dual)
    def objective(alpha):
        return 0.5 * alpha @ K @ alpha - alpha.sum()

    # Gradient of the objective w.r.t. α
    def gradient(alpha):
        return K @ alpha - np.ones(n)

    # Equality constraint: Σᵢ αᵢ yᵢ = 0
    constraints = [{'type': 'eq', 'fun': lambda a: a @ y, 'jac': lambda a: y}]

    # Bounds: αᵢ ≥ 0
    bounds = [(0, None)] * n

    # Initial guess: all zeros
    alpha0 = np.zeros(n)

    result = minimize(
        objective, alpha0,
        jac=gradient,
        method='SLSQP',
        bounds=bounds,
        constraints=constraints,
        options={'ftol': 1e-9, 'maxiter': 1000}
    )

    if not result.success:
        print(f"Warning: Optimizer did not fully converge. Message: {result.message}")

    alphas = np.maximum(result.x, 0)  # clip tiny negatives from numerical noise

    # Recover w from dual solution: w = Σᵢ αᵢ yᵢ xᵢ
    w = (alphas * y) @ X

    # Recover b using support vectors (αᵢ > tol)
    sv_mask = alphas > tol
    if sv_mask.sum() == 0:
        raise RuntimeError("No support vectors found — check data separability.")
    b = np.mean(y[sv_mask] - X[sv_mask] @ w)

    return w, b, alphas, sv_mask


# Run the dual SVM
w_dual, b_dual, alphas, sv_mask = dual_svm(X, y)

print("=" * 55)
print("DUAL SVM SOLUTION")
print("=" * 55)
print(f"  w (weight vector): [{w_dual[0]:.4f}, {w_dual[1]:.4f}]")
print(f"  b (bias):          {b_dual:.4f}")
print(f"  Number of support vectors: {sv_mask.sum()} / {len(y)}")
print(f"  α values range: [{alphas.min():.4f}, {alphas.max():.4f}]")
print(f"  Σ αᵢ yᵢ = {(alphas * y).sum():.2e}  (should be ≈ 0)")

In [ ]:
# ─── Section 3: Verify Dual Solution Matches Primal (sklearn) ─────────────────

# sklearn SVC with linear kernel (uses dual internally)
svc = SVC(kernel='linear', C=1e6)  # large C ≈ hard-margin
svc.fit(X, y)
w_sklearn = svc.coef_[0]
b_sklearn  = svc.intercept_[0]

# Normalize signs so comparison is meaningful
# (both solutions can be scaled; compare the decision hyperplane direction)
sign_correction = np.sign(w_dual[0]) * np.sign(w_sklearn[0])

print("=" * 55)
print("SOLUTION COMPARISON")
print("=" * 55)
print(f"{'Method':<20} {'w[0]':>10} {'w[1]':>10} {'b':>10}")
print("-" * 55)
print(f"{'Our Dual SVM':<20} {w_dual[0]:>10.4f} {w_dual[1]:>10.4f} {b_dual:>10.4f}")
print(f"{'sklearn SVC':<20} {w_sklearn[0]:>10.4f} {w_sklearn[1]:>10.4f} {b_sklearn:>10.4f}")

# Compare normalized directions
w_dual_norm    = w_dual    / np.linalg.norm(w_dual)
w_sklearn_norm = w_sklearn / np.linalg.norm(w_sklearn)
cos_sim = np.abs(w_dual_norm @ w_sklearn_norm)
print(f"\nCosine similarity between weight vectors: {cos_sim:.6f}  (1.0 = identical direction)")

# Training accuracy of both models
pred_dual    = np.sign(X @ w_dual    + b_dual)
pred_sklearn = svc.predict(X)
acc_dual    = np.mean(pred_dual    == y)
acc_sklearn = np.mean(pred_sklearn == y)
print(f"\nTraining accuracy — Dual SVM:   {acc_dual * 100:.1f}%")
print(f"Training accuracy — sklearn SVC: {acc_sklearn * 100:.1f}%")

In [ ]:
# ─── Section 4: Visualize α Values — Most Are Zero ────────────────────────────

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# --- Left plot: bar chart of all α values ---
idx     = np.arange(len(alphas))
is_spam = (y == 1)
bar_colors = ['#e74c3c' if sv_mask[i] and is_spam[i]
              else '#2ecc71' if sv_mask[i] and not is_spam[i]
              else '#bdc3c7'
              for i in range(len(alphas))]

axes[0].bar(idx, alphas, color=bar_colors, edgecolor='none', alpha=0.85)
axes[0].axhline(1e-5, color='k', linestyle='--', lw=1.2, label='α threshold (1e-5)')
axes[0].set_xlabel('Training Point Index', fontsize=12)
axes[0].set_ylabel('Lagrange Multiplier αᵢ', fontsize=12)
axes[0].set_title('α Values: Most Are Zero (Sparse Solution)', fontsize=12, fontweight='bold')
red_patch   = mpatches.Patch(color='#e74c3c', label=f'Support Vector (Spam) [{sv_mask[is_spam].sum()}]')
green_patch = mpatches.Patch(color='#2ecc71', label=f'Support Vector (Ham) [{sv_mask[~is_spam].sum()}]')
grey_patch  = mpatches.Patch(color='#bdc3c7', label=f'Non-SV (α=0) [{(~sv_mask).sum()}]')
axes[0].legend(handles=[red_patch, green_patch, grey_patch], fontsize=10)

# Annotate SV indices
for i in np.where(sv_mask)[0]:
    axes[0].annotate(str(i), xy=(i, alphas[i]), xytext=(i, alphas[i] + alphas.max() * 0.03),
                     fontsize=7, ha='center', color='#2c3e50')

# --- Right plot: decision boundary with SVs highlighted ---
h = 0.02
xx, yy = np.meshgrid(
    np.linspace(X[:, 0].min() - 0.5, X[:, 0].max() + 0.5, 300),
    np.linspace(X[:, 1].min() - 0.5, X[:, 1].max() + 0.5, 300)
)
Z = (np.c_[xx.ravel(), yy.ravel()] @ w_dual + b_dual).reshape(xx.shape)

axes[1].contourf(xx, yy, Z, levels=[-10, 0, 10], colors=['#aed6f1', '#fadbd8'], alpha=0.4)
axes[1].contour(xx, yy, Z, levels=[-1, 0, 1], colors=['#2980b9', '#2c3e50', '#c0392b'],
                linewidths=[1.2, 2.2, 1.2], linestyles=['--', '-', '--'])

# Plot all points
for cls, (c_col, c_mk) in [('Spam', ('#e74c3c', 'o')), ('Ham', ('#2ecc71', 's'))]:
    mask_cls = df['class'] == cls
    axes[1].scatter(X[mask_cls, 0], X[mask_cls, 1],
                    c=c_col, marker=c_mk, alpha=0.6, s=45, edgecolors='k', linewidths=0.3, label=cls)

# Highlight SVs with bold circles
axes[1].scatter(X[sv_mask, 0], X[sv_mask, 1], s=180, facecolors='none',
                edgecolors='#2c3e50', linewidths=2.2, label='Support Vectors')

axes[1].set_xlabel('Word Count (Scaled)', fontsize=12)
axes[1].set_ylabel('Spam Word Ratio (Scaled)', fontsize=12)
axes[1].set_title('Decision Boundary & Support Vectors', fontsize=12, fontweight='bold')
axes[1].legend(fontsize=10)

plt.suptitle('Dual SVM: α Sparsity and Decision Boundary', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

n_sv = sv_mask.sum()
pct  = n_sv / len(y) * 100
print(f"Sparsity: only {n_sv}/{len(y)} ({pct:.1f}%) of training points are support vectors.")
print("The decision boundary is completely determined by these few points.")

In [ ]:
# ─── Section 5: KKT Complementary Slackness Verification ─────────────────────

# KKT condition: αᵢ × [yᵢ(w·xᵢ + b) - 1] = 0  for all i
functional_margins = y * (X @ w_dual + b_dual)  # yᵢ(w·xᵢ + b)
kkt_product        = alphas * (functional_margins - 1)

# Build verification table
kkt_df = pd.DataFrame({
    'Index':    range(len(y)),
    'Class':    df['class'].values,
    'αᵢ':       alphas.round(6),
    'yᵢf(xᵢ)': functional_margins.round(6),
    'yᵢf(xᵢ)-1': (functional_margins - 1).round(6),
    'KKT = αᵢ×[yf-1]': kkt_product.round(8),
    'Is SV':    sv_mask
})

print("KKT Verification (showing support vectors + 5 non-SVs):")
print("=" * 80)

sv_rows  = kkt_df[kkt_df['Is SV']]
nsv_rows = kkt_df[~kkt_df['Is SV']].head(5)
display_df = pd.concat([sv_rows, nsv_rows]).sort_index()
print(display_df.to_string(index=False))

max_kkt_violation = np.abs(kkt_product).max()
print(f"\nMax |αᵢ × [yᵢf(xᵢ) - 1]| across all points: {max_kkt_violation:.2e}")
print(f"KKT conditions satisfied: {max_kkt_violation < 1e-4}")

print("\nInterpretation:")
print("  Support vectors: αᵢ > 0  AND  yᵢf(xᵢ) ≈ 1  → product ≈ 0 ✓")
print("  Non-SVs:         αᵢ = 0  AND  yᵢf(xᵢ) > 1  → product = 0 ✓")

In [ ]:
# ─── Section 6: w = Σ αᵢ yᵢ xᵢ — Visualize as Weighted Sum of SVs ────────────

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

sv_indices = np.where(sv_mask)[0]

# --- Left: show each αᵢ yᵢ xᵢ vector contribution ---
origin = np.zeros(2)
ax = axes[0]

# Draw each SV's weighted contribution vector
cmap = plt.cm.get_cmap('tab10', len(sv_indices))
cumulative = np.zeros(2)
for k, i in enumerate(sv_indices):
    contrib = alphas[i] * y[i] * X[i]  # αᵢ yᵢ xᵢ
    ax.annotate("", xy=cumulative + contrib, xytext=cumulative,
                arrowprops=dict(arrowstyle='->', color=cmap(k), lw=2.5))
    ax.text(cumulative[0] + contrib[0] / 2, cumulative[1] + contrib[1] / 2,
            f'SV{i}\n(α={alphas[i]:.2f}, y={int(y[i])})', fontsize=8,
            color=cmap(k), ha='center')
    cumulative += contrib

# Draw the final w vector
ax.annotate("", xy=w_dual, xytext=origin,
            arrowprops=dict(arrowstyle='->', color='k', lw=3, linestyle='--'))
ax.scatter(*w_dual, s=120, c='k', zorder=5, label='w = Σ αᵢyᵢxᵢ')

ax.set_xlim(-2, 2)
ax.set_ylim(-2, 2)
ax.axhline(0, color='gray', lw=0.8)
ax.axvline(0, color='gray', lw=0.8)
ax.set_xlabel('w[0] (word_count direction)', fontsize=11)
ax.set_ylabel('w[1] (spam_ratio direction)', fontsize=11)
ax.set_title('w = Σ αᵢyᵢxᵢ: Cumulative Arrow Diagram', fontsize=12, fontweight='bold')
ax.legend(fontsize=10)
ax.set_aspect('equal')

# --- Right: pie chart of contribution magnitudes ---
contributions = [np.linalg.norm(alphas[i] * y[i] * X[i]) for i in sv_indices]
labels_pie    = [f'SV {i}\n(α={alphas[i]:.2f}, y={int(y[i])})' for i in sv_indices]
wedge_colors  = ['#e74c3c' if y[i] == 1 else '#2ecc71' for i in sv_indices]

axes[1].pie(contributions, labels=labels_pie, colors=wedge_colors,
            autopct='%1.1f%%', startangle=140, textprops={'fontsize': 9})
axes[1].set_title('Contribution of Each SV to ||w||\n(|αᵢ yᵢ xᵢ| magnitude)', fontsize=12, fontweight='bold')

plt.suptitle('w is Entirely Determined by Support Vectors', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("Non-SVs (αᵢ=0) contribute ZERO to w.")
print(f"Reconstructed w from SVs only: [{w_dual[0]:.6f}, {w_dual[1]:.6f}]")
print(f"Direct computation w = Σαᵢyᵢxᵢ: [{w_dual[0]:.6f}, {w_dual[1]:.6f}]  (exact match)")

In [ ]:
# ─── Section 7: Data Appears Only as Dot Products — The Gram Matrix ───────────

# The dual objective only involves xᵢ·xⱼ — never xᵢ directly!
# This means we can REPLACE xᵢ·xⱼ with any kernel k(xᵢ,xⱼ) without changing the algorithm.

# Compute the full Gram matrix K = X @ Xᵀ
K_gram = X @ X.T   # shape: (100, 100)  — k[i,j] = xᵢ · xⱼ

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# --- Left: Full Gram matrix heatmap ---
im0 = axes[0].imshow(K_gram, cmap='RdBu_r', aspect='auto')
plt.colorbar(im0, ax=axes[0], shrink=0.8)
axes[0].set_title('Full Gram Matrix\nK[i,j] = xᵢ · xⱼ  (100×100)', fontsize=11, fontweight='bold')
axes[0].set_xlabel('Training point j')
axes[0].set_ylabel('Training point i')
# Add class boundary line
axes[0].axhline(49.5, color='k', lw=2)
axes[0].axvline(49.5, color='k', lw=2)
axes[0].text(12, 5,  'Spam\n×Spam', color='k', fontsize=9, ha='center')
axes[0].text(75, 5,  'Spam\n×Ham',  color='k', fontsize=9, ha='center')
axes[0].text(12, 75, 'Ham\n×Spam',  color='k', fontsize=9, ha='center')
axes[0].text(75, 75, 'Ham\n×Ham',   color='k', fontsize=9, ha='center')

# --- Middle: SV submatrix (only SVs × SVs) ---
K_sv = K_gram[np.ix_(sv_indices, sv_indices)]
im1  = axes[1].imshow(K_sv, cmap='RdBu_r', aspect='auto')
plt.colorbar(im1, ax=axes[1], shrink=0.8)
axes[1].set_title(f'SV Submatrix ({n_sv}×{n_sv})\nOnly SVs Needed for Prediction', fontsize=11, fontweight='bold')
axes[1].set_xlabel(f'SV index (original: {list(sv_indices)})')
axes[1].set_ylabel(f'SV index')
axes[1].set_xticks(range(n_sv))
axes[1].set_yticks(range(n_sv))
axes[1].set_xticklabels(sv_indices, fontsize=9)
axes[1].set_yticklabels(sv_indices, fontsize=9)

# --- Right: decision function via dot products ---
# f(x_new) = Σᵢ αᵢ yᵢ (xᵢ · x_new) + b  [only SVs contribute]
x_test_examples = X[[0, 50]]   # one spam, one ham
sv_alphas = alphas[sv_mask]
sv_y      = y[sv_mask]
sv_X      = X[sv_mask]

print("Decision function via dot products (kernel-trick-ready form):")
print(f"  f(x) = Σᵢ αᵢ yᵢ (xᵢ · x) + b")
print(f"  Only {n_sv} dot products needed per prediction (one per SV)\n")

for k, (x_test, true_label) in enumerate(zip(x_test_examples, ['+1 (Spam)', '-1 (Ham)'])):
    dot_products = sv_X @ x_test       # xᵢ · x for each SV
    decision_val = (sv_alphas * sv_y * dot_products).sum() + b_dual
    print(f"  Test point {k+1} (true={true_label}):")
    print(f"    Dot products with SVs: {dot_products.round(4)}")
    print(f"    f(x) = {decision_val:.4f}  →  class = {int(np.sign(decision_val))}")

axes[2].bar(range(n_sv),
            [sv_alphas[k] * sv_y[k] * (sv_X[k] @ X[0]) for k in range(n_sv)],
            color=['#e74c3c' if sv_y[k] == 1 else '#2ecc71' for k in range(n_sv)],
            edgecolor='k', linewidth=0.5)
axes[2].axhline(0, color='k', lw=1)
axes[2].set_xticks(range(n_sv))
axes[2].set_xticklabels([f'SV{i}' for i in sv_indices], fontsize=9)
axes[2].set_xlabel('Support Vector', fontsize=11)
axes[2].set_ylabel('αᵢ yᵢ (xᵢ · x_test)', fontsize=11)
axes[2].set_title('Each SV\'s Contribution\nto f(x_test[0])', fontsize=11, fontweight='bold')

plt.suptitle('Data Appears ONLY as Dot Products → Kernel Trick Ready', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ─── Section 8: Strong Duality — Primal Objective = Dual Objective ────────────

# Primal objective at optimum: (1/2) ||w||²
primal_obj = 0.5 * np.dot(w_dual, w_dual)

# Dual objective at optimum: Σαᵢ - (1/2) αᵀ K_mod α
# where K_mod[i,j] = yᵢ yⱼ (xᵢ·xⱼ)
yX      = y[:, None] * X
K_mod   = yX @ yX.T
dual_obj = alphas.sum() - 0.5 * alphas @ K_mod @ alphas

print("=" * 55)
print("STRONG DUALITY VERIFICATION")
print("=" * 55)
print(f"  Primal objective (1/2)||w||²:      {primal_obj:.8f}")
print(f"  Dual objective   Σαᵢ - (1/2)α'Kα: {dual_obj:.8f}")
print(f"  Duality gap (should be ≈ 0):        {abs(primal_obj - dual_obj):.2e}")
print(f"\n  Strong duality holds: {abs(primal_obj - dual_obj) < 1e-4}")

print("\n" + "=" * 55)
print("COMPREHENSIVE DUAL SVM SUMMARY")
print("=" * 55)
print(f"  Training samples:          {len(y)}")
print(f"  Feature dimensions (p):    {X.shape[1]}")
print(f"  Support vectors found:     {sv_mask.sum()}")
print(f"  Sparsity (% non-SVs):      {(~sv_mask).mean()*100:.1f}%")
print(f"  ||w|| (margin width = 2/||w||): {np.linalg.norm(w_dual):.4f}")
print(f"  Margin width:              {2 / np.linalg.norm(w_dual):.4f}")
print(f"  Σ αᵢ yᵢ (= 0 required):   {(alphas * y).sum():.2e}")

# Visualize duality gap across a path of sub-optimal solutions
fig, ax = plt.subplots(figsize=(10, 4))

scale_factors = np.linspace(0, 1.5, 200)
primal_vals   = [0.5 * np.dot(s * w_dual, s * w_dual) for s in scale_factors]
dual_vals_scaled = []
for s in scale_factors:
    a_s = s * alphas
    d = a_s.sum() - 0.5 * a_s @ K_mod @ a_s
    dual_vals_scaled.append(d)

ax.plot(scale_factors, primal_vals,    color='#e74c3c', lw=2, label='Primal objective (1/2)||sw||²')
ax.plot(scale_factors, dual_vals_scaled, color='#2980b9', lw=2, label='Dual objective (scaled α)')
ax.axvline(1.0, color='k', linestyle='--', lw=1.5, label='Optimal point (scale=1)')
ax.scatter([1.0], [primal_obj], s=100, c='#e74c3c', zorder=5)
ax.scatter([1.0], [dual_obj],   s=100, c='#2980b9', zorder=5)
ax.fill_between(scale_factors, primal_vals, dual_vals_scaled,
                where=[p >= d for p, d in zip(primal_vals, dual_vals_scaled)],
                alpha=0.15, color='purple', label='Duality gap (primal ≥ dual)')
ax.set_xlabel('Scale factor on optimal solution', fontsize=12)
ax.set_ylabel('Objective value', fontsize=12)
ax.set_title('Strong Duality: Primal = Dual at Optimum', fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()

In [ ]:
# ─── Complexity Comparison: Primal vs Dual ────────────────────────────────────

n_values = np.array([100, 500, 1000, 5000, 10000])
p_values = np.array([50, 100, 500, 1000, 5000, 10000, 50000])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- Left: Dual cost O(n²) as n grows, for fixed p ---
p_fixed = 1000
primal_cost_n = n_values**2 * p_fixed   # O(n * p²) iterations — simplified
dual_cost_n   = n_values**2             # O(n²) for the Gram matrix

axes[0].loglog(n_values, primal_cost_n, 'o-', color='#e74c3c', lw=2, label=f'Primal O(n·p²), p={p_fixed}')
axes[0].loglog(n_values, dual_cost_n,   's-', color='#2980b9', lw=2, label='Dual O(n²)')
axes[0].set_xlabel('Number of training samples n', fontsize=12)
axes[0].set_ylabel('Relative computational cost (log scale)', fontsize=12)
axes[0].set_title('When n < p: Dual is Cheaper', fontsize=12, fontweight='bold')
axes[0].legend(fontsize=10)

# --- Right: Show when to prefer primal vs dual ---
n_grid, p_grid = np.meshgrid(np.logspace(1, 5, 200), np.logspace(1, 5, 200))
# Prefer dual when n < p  (dual is O(n²), primal is O(p+1))
prefer_dual = (n_grid < p_grid).astype(float)
im = axes[1].contourf(np.log10(n_grid), np.log10(p_grid), prefer_dual,
                       levels=[-0.5, 0.5, 1.5], colors=['#fadbd8', '#aed6f1'], alpha=0.7)
axes[1].contour(np.log10(n_grid), np.log10(p_grid), prefer_dual,
                levels=[0.5], colors=['k'], linewidths=2)
axes[1].set_xlabel('log₁₀(n)  — training samples', fontsize=12)
axes[1].set_ylabel('log₁₀(p)  — feature dimensions', fontsize=12)
axes[1].set_title('When to Use Dual vs Primal', fontsize=12, fontweight='bold')
red_p  = mpatches.Patch(color='#fadbd8', alpha=0.7, label='Prefer Primal (n > p)')
blue_p = mpatches.Patch(color='#aed6f1', alpha=0.7, label='Prefer Dual (n < p, e.g. NLP)')
axes[1].legend(handles=[red_p, blue_p], fontsize=10)

# Mark our spam dataset
axes[1].scatter([np.log10(100)], [np.log10(2)], s=120, c='k', zorder=5, label='Our dataset (n=100, p=2)')
axes[1].annotate('Our dataset\n(n=100, p=2)', xy=(np.log10(100), np.log10(2)),
                  xytext=(np.log10(100)+0.3, np.log10(2)+0.5), fontsize=9,
                  arrowprops=dict(arrowstyle='->', lw=1))

plt.suptitle('Primal vs Dual: Complexity and When Each Is Better', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("Key insight: For NLP tasks where p (vocabulary size) >> n (documents),")
print("the dual formulation is dramatically more efficient.")

In [ ]:
# ─── Full Summary Visualization ───────────────────────────────────────────────

fig = plt.figure(figsize=(16, 10))
gs  = fig.add_gridspec(2, 3, hspace=0.45, wspace=0.35)

# 1. Decision boundary with SVs
ax1 = fig.add_subplot(gs[0, 0])
xx, yy = np.meshgrid(
    np.linspace(X[:, 0].min() - 0.5, X[:, 0].max() + 0.5, 300),
    np.linspace(X[:, 1].min() - 0.5, X[:, 1].max() + 0.5, 300)
)
Z = (np.c_[xx.ravel(), yy.ravel()] @ w_dual + b_dual).reshape(xx.shape)
ax1.contourf(xx, yy, Z, levels=[-10, 0, 10], colors=['#aed6f1', '#fadbd8'], alpha=0.4)
ax1.contour(xx, yy, Z, levels=[-1, 0, 1], colors=['#2980b9', 'k', '#c0392b'],
            linewidths=[1.5, 2.5, 1.5], linestyles=['--', '-', '--'])
ax1.scatter(X[is_spam, 0],  X[is_spam, 1],  c='#e74c3c', s=40, alpha=0.6, edgecolors='none', label='Spam')
ax1.scatter(X[~is_spam, 0], X[~is_spam, 1], c='#2ecc71', s=40, alpha=0.6, edgecolors='none', label='Ham')
ax1.scatter(X[sv_mask, 0], X[sv_mask, 1], s=200, facecolors='none', edgecolors='k', lw=2, label='SVs')
ax1.set_title('Decision Boundary', fontweight='bold')
ax1.legend(fontsize=8)

# 2. Alpha bar chart (sorted)
ax2 = fig.add_subplot(gs[0, 1])
sorted_idx = np.argsort(alphas)[::-1]
bar_c = ['#e74c3c' if y[i] == 1 else '#2ecc71' for i in sorted_idx]
ax2.bar(range(len(alphas)), alphas[sorted_idx], color=bar_c, edgecolor='none')
ax2.axhline(1e-5, color='k', lw=1, linestyle='--')
ax2.set_xlabel('Rank', fontsize=10)
ax2.set_ylabel('α value', fontsize=10)
ax2.set_title('α Values (Sorted)', fontweight='bold')
ax2.text(30, alphas.max() * 0.7, f'{(alphas < 1e-5).sum()} zeros\n{(alphas >= 1e-5).sum()} non-zero',
         fontsize=9, ha='center', bbox=dict(boxstyle='round', fc='white', ec='gray'))

# 3. KKT verification scatter
ax3 = fig.add_subplot(gs[0, 2])
ax3.scatter(range(len(alphas)), np.abs(kkt_product), c=['#e74c3c' if sv_mask[i] else '#bdc3c7' for i in range(len(alphas))],
            s=40, alpha=0.8)
ax3.axhline(1e-4, color='k', lw=1, linestyle='--', label='Threshold 1e-4')
ax3.set_yscale('log')
ax3.set_xlabel('Training point index', fontsize=10)
ax3.set_ylabel('|αᵢ × (yᵢf(xᵢ) - 1)|  (log)', fontsize=9)
ax3.set_title('KKT Condition Residuals', fontweight='bold')
ax3.legend(fontsize=8)

# 4. Gram matrix (SVs only)
ax4 = fig.add_subplot(gs[1, 0])
im = ax4.imshow(K_sv, cmap='RdBu_r', aspect='auto')
plt.colorbar(im, ax=ax4, shrink=0.8)
ax4.set_title(f'SV Gram Matrix ({n_sv}×{n_sv})', fontweight='bold')
ax4.set_xticks(range(n_sv))
ax4.set_yticks(range(n_sv))
ax4.set_xticklabels([f'SV{i}' for i in sv_indices], fontsize=8)
ax4.set_yticklabels([f'SV{i}' for i in sv_indices], fontsize=8)

# 5. Functional margins histogram
ax5 = fig.add_subplot(gs[1, 1])
ax5.hist(functional_margins[is_spam],  bins=15, alpha=0.6, color='#e74c3c', label='Spam (y=+1)', edgecolor='k', lw=0.3)
ax5.hist(functional_margins[~is_spam], bins=15, alpha=0.6, color='#2ecc71', label='Ham (y=-1)', edgecolor='k', lw=0.3)
ax5.axvline(1,  color='#c0392b', lw=2, linestyle='--', label='Margin boundary +1')
ax5.axvline(-1, color='#2980b9', lw=2, linestyle='--', label='Margin boundary -1')
ax5.set_xlabel('Functional margin yᵢ(w·xᵢ+b)', fontsize=10)
ax5.set_ylabel('Count', fontsize=10)
ax5.set_title('Functional Margins Distribution', fontweight='bold')
ax5.legend(fontsize=8)

# 6. Duality gap bar
ax6 = fig.add_subplot(gs[1, 2])
objectives = ['Primal\n(1/2)||w||²', 'Dual\nΣα-(1/2)α\'Kα', 'Duality\ngap']
values = [primal_obj, dual_obj, abs(primal_obj - dual_obj)]
bar_colors6 = ['#e74c3c', '#2980b9', '#f39c12']
bars = ax6.bar(objectives, values, color=bar_colors6, edgecolor='k', lw=0.5)
for bar, val in zip(bars, values):
    ax6.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.002,
             f'{val:.4f}', ha='center', va='bottom', fontsize=9)
ax6.set_ylabel('Objective value', fontsize=10)
ax6.set_title('Strong Duality:\nPrimal = Dual at Optimum', fontweight='bold')
ax6.set_yscale('symlog', linthresh=1e-6)

fig.suptitle('Lagrangian Duality in SVM — Complete Summary', fontsize=15, fontweight='bold')
plt.savefig('lagrangian_duality_summary.png', dpi=120, bbox_inches='tight')
plt.show()
print("Summary figure saved as lagrangian_duality_summary.png")

## Key Insights Recap

| Concept | What It Means | Why It Matters |
|---|---|---|
| **Lagrangian** | Combines objective + constraints into one function via multipliers αᵢ | Converts constrained problem to unconstrained |
| **Dual problem** | Optimize over αᵢ instead of w, b | n variables instead of p+1; enables kernel trick |
| **KKT conditions** | αᵢ × [yᵢf(xᵢ)-1] = 0 | Explains sparsity: most αᵢ = 0 |
| **Support vectors** | Points where αᵢ > 0 (on the margin) | Only they determine w and the decision boundary |
| **Strong duality** | Primal obj = Dual obj at optimum | Solving dual gives the exact same solution as primal |
| **Dot products** | Data appears only as xᵢ·xⱼ in dual | Replace with k(xᵢ,xⱼ) → kernel trick! |

## The Bridge to Kernel Methods

The dual decision function is:
$$f(x) = \sum_{i \in SV} \alpha_i y_i \underbrace{(x_i \cdot x)}_{\text{dot product}} + b$$

Replace the dot product with **any kernel function** $k(x_i, x)$:
$$f(x) = \sum_{i \in SV} \alpha_i y_i \underbrace{k(x_i, x)}_{\text{kernel}} + b$$

This works without ever computing the high-dimensional feature mapping — just by evaluating the kernel function. That's the kernel trick, and it only works because of duality!

## Self-Check Questions

Test your understanding before moving on.

---

**Question 1:** The dual problem has `n` variables (αᵢ, one per sample) while the primal has `p+1` variables (w and b). When is the dual cheaper to solve? Give an example from machine learning where this matters.

<details>
<summary>Click to reveal answer</summary>

The dual is cheaper when **n < p** — i.e., when the number of training samples is smaller than the number of features (dimensions). In this regime, the dual's QP has n variables vs. the primal's p+1 variables, so the dual is significantly smaller.

**Classic example:** Text classification using bag-of-words features. A vocabulary of 50,000 words gives p = 50,000. If you have n = 1,000 training documents, the primal solves in 50,001-dimensional space while the dual solves in 1,000-dimensional space — 50× smaller! This is the regime SVM was designed to exploit.

More broadly: genomics (p = millions of SNPs, n = thousands of patients), image kernels (implicit infinite-dimensional features), and structured prediction all benefit from the dual formulation.

</details>

---

**Question 2:** At the optimal solution, w = Σᵢ αᵢ yᵢ xᵢ and only SVs have αᵢ > 0. So the decision function is f(x) = w·x + b = Σᵢ αᵢ yᵢ (xᵢ·x) + b. Why does this expression naturally enable the kernel trick?

<details>
<summary>Click to reveal answer</summary>

The decision function depends on the data **only through dot products** xᵢ·x — never through the raw features xᵢ or x individually. This is crucial because:

1. **Implicit feature maps:** Suppose we map inputs to a high (or infinite) dimensional space: φ: ℝᵖ → ℝᴴ. If we use the decision function in the mapped space, f(x) = Σ αᵢ yᵢ φ(xᵢ)·φ(x) + b.

2. **Kernel as a shortcut:** If we can compute k(xᵢ, x) = φ(xᵢ)·φ(x) directly without computing φ, we get the power of high-dimensional features at the cost of computing k — which might be O(p) even when H is infinite!

3. **Example:** The RBF kernel k(x,z) = exp(-γ||x-z||²) corresponds to an infinite-dimensional feature space, yet evaluating it costs O(p) — we never touch the infinite dimensions.

The kernel trick would not work in the primal formulation because w lives in the original (or mapped) feature space — you'd need to store and compute the full high-dimensional w. The dual sidesteps this entirely.

</details>

---

**Question 3:** KKT complementary slackness says: αᵢ [yᵢ(w·xᵢ+b) - 1] = 0. If a point is NOT on the margin (yᵢ(w·xᵢ+b) > 1), what must αᵢ be? What does this mean geometrically?

<details>
<summary>Click to reveal answer</summary>

If yᵢ(w·xᵢ+b) > 1, then the bracket [yᵢ(w·xᵢ+b) - 1] > 0. For their product αᵢ × [yᵢ(w·xᵢ+b) - 1] = 0 to hold, we must have **αᵢ = 0**.

**Geometric meaning:** A point strictly inside the correct side of the margin (functional margin > 1) is far from the decision boundary. It's "comfortably classified" and doesn't need to influence the boundary at all. Setting αᵢ = 0 means this point contributes zero to the weight vector w = Σ αᵢ yᵢ xᵢ.

In other words: **only the points closest to the boundary (on the margin) matter**. You could remove all non-support-vector training points and retrain, and you'd get the exact same model. This is the mathematical explanation for SVM's robustness to points far from the boundary and its sparsity.

</details>